<a href="https://colab.research.google.com/github/akshayanandraut/colab/blob/main/notebooks/comfyui_colab_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# ComfyUI Photorealistic Indian Influencer Setup
1. Run the Environment Setup.
2. Download the required models.
3. Enter your ngrok token and launch.
```

In [6]:
#@title 1. Environment Setup
import os

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
else:
    WORKSPACE = "/content/ComfyUI"

if not os.path.exists(WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}

%cd {WORKSPACE}
!pip install pyngrok xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ComfyUI
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121


In [8]:
#@title 2. Verify Models & Cleanup
import os

%cd {WORKSPACE}
os.makedirs('./models/checkpoints/', exist_ok=True)
os.makedirs('./models/upscale_models/', exist_ok=True)

ckpt_path = './models/checkpoints/realvisxlV50_v50Bakedvae.safetensors'

# Automatic cleanup of corrupt placeholders
if os.path.exists(ckpt_path):
    size = os.path.getsize(ckpt_path)
    if size < 1000000: # Less than 1MB is definitely a failed download
        print(f"☑️ Removing corrupt file placeholder ({size} bytes)...")
        os.remove(ckpt_path)
        print("✅ Done. Please upload the full 6.5GB model to /ComfyUI/models/checkpoints/ now.")
    else:
        print(f"✅ RealVisXL V5.0 detected ({size / (1024**3):.2f} GB).")
else:
    print("⌛ RealVisXL not found. Waiting for manual upload to: ComfyUI/models/checkpoints/")

# 4x-UltraSharp
print("\nChecking 4x-UltraSharp...")
!wget -c -L "https://huggingface.co/lokCX/4x-UltraSharp/resolve/main/4x-UltraSharp.pth" -P ./models/upscale_models/

/content/drive/MyDrive/ComfyUI
⌛ RealVisXL not found. Waiting for manual upload to: ComfyUI/models/checkpoints/

Checking 4x-UltraSharp...
--2026-07-31 10:21:58--  https://huggingface.co/lokCX/4x-UltraSharp/resolve/main/4x-UltraSharp.pth
Resolving huggingface.co (huggingface.co)... 13.35.202.97, 13.35.202.121, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.97|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth [following]
--2026-07-31 10:21:58--  https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/6430096543a53c86b3fcb2a0/080be486975ca8916a1a6fda9e763332b218dbc306ca993f2a029595919e72be?response-content-disposition=inline%3B+filename*%3DUTF-8%27%274x-UltraSharp.pth%3B+filename%3D%224x-UltraSharp.pth%2

In [ ]:
#@title 3. Launch ComfyUI via Ngrok (with Real-time Logs)
from pyngrok import ngrok
import threading
import subprocess
import sys
import os

# Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "3HG2NXigok7WNH2YMXhlymV3M1j_2YP3SwKDSroFHjKj87L3B" #@param {type:"string"}
PORT = 8188

def start_ngrok(port):
    if not NGROK_AUTH_TOKEN:
        print("❌ ERROR: Please enter your ngrok token!")
        return
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    ngrok.kill()
    url = ngrok.connect(port, "http").public_url
    print(f"\n[Ready] URL: {url}\n")

def run_comfy():
    os.chdir(WORKSPACE)
    # Bound to 0.0.0.0 for ngrok compatibility and explicitly set cuda-device 0
    process = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(PORT), "--enable-cors-header", "--force-fp16", "--highvram", "--cuda-device", "0"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end="")

# Start tunnel
if NGROK_AUTH_TOKEN:
    threading.Thread(target=start_ngrok, args=(PORT,), daemon=True).start()

# Start ComfyUI with logging
run_comfy()

Exception in thread Thread-11 (start_ngrok):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pyngrok/ngrok.py", line 632, in api_request
    response = urlopen(request, encoded_data, timeout)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 215, in urlopen
    return opener.open(url, data, timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 521, in open
    response = meth(req, response)
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 630, in http_response
    response = self.parent.error(
               ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 559, in error
    return self._call_chain(*args)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 492, in _call_chain
    result = func(*args)
             ^^^^^^^^^^^
  File "/usr/lib/python3

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] Set cuda device to: 0
[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
[INFO] Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'int8_linear', 'quantize_and_rotate_rowwise', 'quantize_int8_rowwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'rms_adaln', 'rms_rope', 'rms_rope1', 'rms_rope1_', 'rms_rope_', 'rms_rope_split_half', 'rms_rope_spl